<a href="https://colab.research.google.com/github/korkutanapa/DCASE2025_TASK2/blob/main/ORJ_DCASE_AE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# DCASE 2025 TASK 2 STYLE NORMAL-ONLY AE SCORING
#
# Input:
#   *_thr.xlsx  -> normal training features
#   other .xlsx -> unlabeled test features
#
# Output per machine:
#   anomaly_score_{MachineType}_section_00_test.csv
#   decision_result_{MachineType}_section_00_test.csv
#
# No headers in output CSV files.
#
# Method:
#   Normal-only AutoEncoder ensemble
#   Scores:
#       latent kNN percentile
#       top-k normal-calibrated z-error percentile
#       z-error percentile
#       MSE percentile
#   Final anomaly score:
#       rank/percentile ensemble
# ============================================================


# ============================================================
# 0. INSTALLS
# ============================================================

!pip install -q numpy pandas scipy scikit-learn matplotlib openpyxl tensorflow keras joblib


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import glob
import random
import zipfile
import warnings

import numpy as np
import pandas as pd

from scipy.stats import rankdata

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.covariance import LedoitWolf

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers

warnings.filterwarnings("ignore")


# ============================================================
# 2. HARDWARE CHECK
# ============================================================

print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")

if len(gpus) > 0:
    print("GPU available:")
    for gpu in gpus:
        print(gpu)
else:
    print("No GPU found. CPU also works, but GPU is recommended.")


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


# ============================================================
# 4. PATH SETTINGS
# ============================================================

# In Google Colab, upload all Excel files to /content
INPUT_DIR = "/content"

# If running inside ChatGPT sandbox, use:
# INPUT_DIR = "/mnt/data"

OUTPUT_DIR = "/content/ae_dcase_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 5. MACHINE TYPES
# ============================================================

MACHINE_TYPES = [
    "AutoTrash",
    "BandSealer",
    "CoffeeGrinder",
    "HomeCamera",
    "Polisher",
    "ScrewFeeder",
    "ToyPet",
    "ToyRCCar",
]


# ============================================================
# 6. GENERAL SETTINGS
# ============================================================

LABEL_COL = "label"

NORMAL_LABEL = "normal"
ANOMALY_LABEL = "anomaly"

EXCLUDE_COLS = [
    "label",
    "file_id",
    "file_path",
    "filename",
    "path",
    "machine",
    "section",
    "domain",
    "target",
    "source",
]

# Normal train / validation split from *_thr file
NORMAL_TRAIN_SIZE = 0.80

# Output decision threshold
# ROC-AUC is not relevant here because evaluation labels are unavailable.
# This threshold is estimated only from normal validation ensemble scores.
THRESHOLD_QUANTILE = 0.95

# AE training
BATCH_SIZE = 32
EPOCHS = 300
PATIENCE = 30

SCALER_TYPE = "standard"  # "standard" or "robust"

# Latent-space kNN
LATENT_K = 10

# Top-k reconstruction deviation
TOPK_FEATURE_RATIO = 0.10

EPS = 1e-8


# ============================================================
# 7. AE MODEL CONFIGURATION
# ============================================================

# Chosen based on previous dev_bearing/dev_fan observations:
# - bearing liked compact latent kNN, latent_dim=4
# - fan liked top-k z-error with larger latent dimension
# Therefore we use a small ensemble to avoid relying on one machine-specific behavior.

AE_CONFIGS = [
    {
        "name": "compact_sparse_latent4",
        "latent_dim": 4,
        "noise_std": 0.0,
        "sparse_l1": 1e-5,
    },
    {
        "name": "medium_denoising_latent8",
        "latent_dim": 8,
        "noise_std": 0.03,
        "sparse_l1": 0.0,
    },
    {
        "name": "wide_sparse_latent32",
        "latent_dim": 32,
        "noise_std": 0.0,
        "sparse_l1": 1e-5,
    },
]


# ============================================================
# 8. HELPER FUNCTIONS
# ============================================================

def find_train_test_files(machine, input_dir):
    """
    Finds:
        train file: contains machine name and '_thr'
        test file : contains machine name but not '_thr'
    """

    all_xlsx = glob.glob(os.path.join(input_dir, "*.xlsx"))

    machine_files = [
        p for p in all_xlsx
        if machine.lower() in os.path.basename(p).lower()
    ]

    train_candidates = [
        p for p in machine_files
        if "_thr" in os.path.basename(p).lower()
    ]

    test_candidates = [
        p for p in machine_files
        if "_thr" not in os.path.basename(p).lower()
    ]

    if len(train_candidates) == 0:
        raise FileNotFoundError(f"No train *_thr file found for {machine}")

    if len(test_candidates) == 0:
        raise FileNotFoundError(f"No test file found for {machine}")

    # Use shortest path name if duplicates exist
    train_file = sorted(train_candidates, key=lambda x: len(os.path.basename(x)))[0]
    test_file = sorted(test_candidates, key=lambda x: len(os.path.basename(x)))[0]

    return train_file, test_file


def encode_labels(label_series):
    labels = label_series.astype(str).str.lower().str.strip()

    y = labels.map({
        NORMAL_LABEL: 0,
        ANOMALY_LABEL: 1,
        "0": 0,
        "1": 1,
    })

    if y.isna().any():
        # If unexpected labels exist, return None instead of crashing.
        return None

    return y.astype(int).values


def get_file_ids(df, n_rows):
    """
    Returns file ids for DCASE-style output.
    Priority:
        file_id
        filename
        basename(file_path)
        basename(path)
        generated section_00_XXXX.wav
    """

    for col in ["file_id", "filename"]:
        if col in df.columns:
            ids = df[col].astype(str).values
            return ids

    for col in ["file_path", "path"]:
        if col in df.columns:
            ids = df[col].astype(str).apply(lambda x: os.path.basename(x)).values
            return ids

    ids = np.array([
        f"section_00_{i:04d}.wav"
        for i in range(n_rows)
    ])

    return ids


def get_common_numeric_features(train_df, test_df):
    train_numeric = train_df.select_dtypes(include=[np.number]).columns.tolist()
    test_numeric = test_df.select_dtypes(include=[np.number]).columns.tolist()

    common_numeric = sorted(list(set(train_numeric).intersection(set(test_numeric))))

    feature_cols = [
        c for c in common_numeric
        if c not in EXCLUDE_COLS
    ]

    valid_features = []

    for col in feature_cols:
        values = train_df[col].replace([np.inf, -np.inf], np.nan)
        if values.nunique(dropna=True) > 1:
            valid_features.append(col)

    return valid_features


def prepare_scaler():
    if SCALER_TYPE == "standard":
        return StandardScaler()
    elif SCALER_TYPE == "robust":
        return RobustScaler()
    else:
        raise ValueError("SCALER_TYPE must be 'standard' or 'robust'.")


def build_autoencoder(input_dim, latent_dim, noise_std, sparse_l1):
    """
    Sparse undercomplete AE.

    Structure:
        input -> 128 -> 64 -> latent -> 64 -> 128 -> output

    This is intentionally not too large.
    """

    inp = layers.Input(shape=(input_dim,), name="input_features")

    x = inp

    if noise_std > 0:
        x = layers.GaussianNoise(noise_std)(x)

    # Encoder
    x = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.20)(x)

    x = layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.15)(x)

    latent = layers.Dense(
        latent_dim,
        activation="relu",
        activity_regularizer=regularizers.l1(sparse_l1),
        name="latent_space"
    )(x)

    # Decoder
    x = layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(latent)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.BatchNormalization()(x)

    out = layers.Dense(
        input_dim,
        activation="linear",
        name="reconstructed_features"
    )(x)

    autoencoder = models.Model(inp, out)
    encoder = models.Model(inp, latent)

    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse"
    )

    return autoencoder, encoder


def reconstruction_error_matrices(autoencoder, X):
    X_hat = autoencoder.predict(X, verbose=0)

    sq_err = (X - X_hat) ** 2
    abs_err = np.abs(X - X_hat)

    return sq_err, abs_err


def normal_reliability_statistics(val_sq_err):
    """
    Normal-only feature-wise reconstruction calibration.
    """

    med = np.median(val_sq_err, axis=0)
    mad = np.median(np.abs(val_sq_err - med), axis=0)
    scale = 1.4826 * mad + EPS

    reliability = 1.0 / (med + scale + EPS)

    lo = np.percentile(reliability, 5)
    hi = np.percentile(reliability, 95)

    reliability = np.clip(reliability, lo, hi)

    weights = reliability / np.mean(reliability)

    return med, scale, weights


def empirical_percentile(reference_scores, scores):
    """
    Converts score values to percentiles relative to normal validation scores.
    Higher percentile = more anomalous.
    """

    ref_sorted = np.sort(reference_scores)

    pct = np.searchsorted(
        ref_sorted,
        scores,
        side="right"
    ) / len(ref_sorted)

    return pct


def reconstruction_score_components(
    sq_err,
    abs_err,
    val_sq_median,
    val_sq_scale,
    weights
):
    """
    Computes reconstruction-based anomaly scores.
    All scores are higher = more anomalous.
    """

    n_features = sq_err.shape[1]
    top_k = max(5, int(TOPK_FEATURE_RATIO * n_features))

    weight_sum = np.sum(weights)

    mse = np.mean(sq_err, axis=1)

    z_err = (sq_err - val_sq_median) / val_sq_scale
    z_pos = np.maximum(z_err, 0.0)

    zmean = np.mean(z_pos, axis=1)

    weighted_zmean = np.sum(z_pos * weights, axis=1) / weight_sum

    sorted_z = np.sort(z_pos, axis=1)
    topk_zmean = np.mean(sorted_z[:, -top_k:], axis=1)

    return {
        "mse": mse,
        "zmean": zmean,
        "weighted_zmean": weighted_zmean,
        "topk_zmean": topk_zmean,
    }


def latent_score_components(encoder, X_ref_normal, X_eval):
    """
    Latent-space anomaly scores.
    Reference is normal training latent vectors only.
    """

    Z_ref = encoder.predict(X_ref_normal, verbose=0)
    Z_eval = encoder.predict(X_eval, verbose=0)

    # latent kNN
    k = min(LATENT_K, len(Z_ref))

    knn = NearestNeighbors(
        n_neighbors=k,
        metric="euclidean"
    )

    knn.fit(Z_ref)

    dists, _ = knn.kneighbors(Z_eval)
    latent_knn = dists.mean(axis=1)

    # latent Mahalanobis
    try:
        cov = LedoitWolf().fit(Z_ref)
        latent_mahal = cov.mahalanobis(Z_eval)
    except Exception:
        latent_mahal = np.zeros(len(Z_eval))

    return {
        "latent_knn": latent_knn,
        "latent_mahalanobis": latent_mahal,
    }


def train_one_ae_and_score(
    config,
    X_normal_train,
    X_normal_val,
    X_test
):
    """
    Trains one AE config and returns validation/test percentile scores.
    """

    tf.keras.backend.clear_session()

    autoencoder, encoder = build_autoencoder(
        input_dim=X_normal_train.shape[1],
        latent_dim=config["latent_dim"],
        noise_std=config["noise_std"],
        sparse_l1=config["sparse_l1"]
    )

    early_stop = callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=PATIENCE,
        restore_best_weights=True
    )

    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=10,
        min_lr=1e-5
    )

    history = autoencoder.fit(
        X_normal_train,
        X_normal_train,
        validation_data=(X_normal_val, X_normal_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop, reduce_lr],
        verbose=0
    )

    # Reconstruction errors
    val_sq, val_abs = reconstruction_error_matrices(autoencoder, X_normal_val)
    test_sq, test_abs = reconstruction_error_matrices(autoencoder, X_test)

    val_median, val_scale, weights = normal_reliability_statistics(val_sq)

    val_recon_scores = reconstruction_score_components(
        sq_err=val_sq,
        abs_err=val_abs,
        val_sq_median=val_median,
        val_sq_scale=val_scale,
        weights=weights
    )

    test_recon_scores = reconstruction_score_components(
        sq_err=test_sq,
        abs_err=test_abs,
        val_sq_median=val_median,
        val_sq_scale=val_scale,
        weights=weights
    )

    # Latent scores
    val_latent_scores = latent_score_components(
        encoder=encoder,
        X_ref_normal=X_normal_train,
        X_eval=X_normal_val
    )

    test_latent_scores = latent_score_components(
        encoder=encoder,
        X_ref_normal=X_normal_train,
        X_eval=X_test
    )

    val_scores_raw = {}
    test_scores_raw = {}

    val_scores_raw.update(val_recon_scores)
    val_scores_raw.update(val_latent_scores)

    test_scores_raw.update(test_recon_scores)
    test_scores_raw.update(test_latent_scores)

    # Select only robust score components for ensemble
    selected_components = [
        "latent_knn",
        "latent_mahalanobis",
        "topk_zmean",
        "zmean",
        "mse",
    ]

    val_pct_components = {}
    test_pct_components = {}

    for score_name in selected_components:
        val_raw = val_scores_raw[score_name]
        test_raw = test_scores_raw[score_name]

        val_pct = empirical_percentile(val_raw, val_raw)
        test_pct = empirical_percentile(val_raw, test_raw)

        full_name = f"{config['name']}__{score_name}_pct"

        val_pct_components[full_name] = val_pct
        test_pct_components[full_name] = test_pct

    info = {
        "config_name": config["name"],
        "latent_dim": config["latent_dim"],
        "noise_std": config["noise_std"],
        "sparse_l1": config["sparse_l1"],
        "best_val_loss": float(np.min(history.history["val_loss"])),
        "n_epochs": len(history.history["loss"])
    }

    return val_pct_components, test_pct_components, info


def create_final_scores(val_component_dict, test_component_dict):
    """
    Rank/percentile ensemble.
    Components are already percentile-calibrated by normal validation.
    """

    val_matrix = np.vstack([
        val_component_dict[k]
        for k in sorted(val_component_dict.keys())
    ])

    test_matrix = np.vstack([
        test_component_dict[k]
        for k in sorted(test_component_dict.keys())
    ])

    val_final = np.mean(val_matrix, axis=0)
    test_final = np.mean(test_matrix, axis=0)

    return val_final, test_final


# ============================================================
# 9. MAIN LOOP
# ============================================================

summary_rows = []

for machine in MACHINE_TYPES:

    print("\n" + "=" * 100)
    print(f"MACHINE: {machine}")
    print("=" * 100)

    train_file, test_file = find_train_test_files(machine, INPUT_DIR)

    print("Train file:", train_file)
    print("Test file :", test_file)

    train_df = pd.read_excel(train_file)
    test_df = pd.read_excel(test_file)

    file_ids = get_file_ids(test_df, len(test_df))

    feature_cols = get_common_numeric_features(train_df, test_df)

    print("Train shape:", train_df.shape)
    print("Test shape :", test_df.shape)
    print("Feature count:", len(feature_cols))

    X_train_raw_all = train_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    X_test_raw = test_df[feature_cols].replace([np.inf, -np.inf], np.nan)

    # If train labels exist, keep only normal rows.
    # If not, assume all *_thr rows are normal training data.
    if LABEL_COL in train_df.columns:
        y_train = encode_labels(train_df[LABEL_COL])
        if y_train is not None:
            X_train_raw_all = X_train_raw_all.iloc[y_train == 0].copy()

    print("Normal training pool:", X_train_raw_all.shape)

    # Normal train / normal validation split
    X_normal_train_raw, X_normal_val_raw = train_test_split(
        X_train_raw_all,
        train_size=NORMAL_TRAIN_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    print("AE normal train:", X_normal_train_raw.shape)
    print("AE normal val  :", X_normal_val_raw.shape)

    # Preprocessing fitted only on normal train
    imputer = SimpleImputer(strategy="median")

    X_normal_train_imp = imputer.fit_transform(X_normal_train_raw)
    X_normal_val_imp = imputer.transform(X_normal_val_raw)
    X_test_imp = imputer.transform(X_test_raw)

    scaler = prepare_scaler()

    X_normal_train = scaler.fit_transform(X_normal_train_imp)
    X_normal_val = scaler.transform(X_normal_val_imp)
    X_test = scaler.transform(X_test_imp)

    # Train AE ensemble
    all_val_components = {}
    all_test_components = {}
    model_infos = []

    for config in AE_CONFIGS:

        print(
            f"Training {config['name']} | "
            f"latent={config['latent_dim']} | "
            f"noise={config['noise_std']} | "
            f"sparse_l1={config['sparse_l1']}"
        )

        val_components, test_components, info = train_one_ae_and_score(
            config=config,
            X_normal_train=X_normal_train,
            X_normal_val=X_normal_val,
            X_test=X_test
        )

        all_val_components.update(val_components)
        all_test_components.update(test_components)
        model_infos.append(info)

        print(
            f"  best_val_loss={info['best_val_loss']:.6f}, "
            f"epochs={info['n_epochs']}"
        )

    # Final ensemble anomaly score
    val_final_score, test_final_score = create_final_scores(
        val_component_dict=all_val_components,
        test_component_dict=all_test_components
    )

    threshold = np.quantile(val_final_score, THRESHOLD_QUANTILE)

    decision = (test_final_score >= threshold).astype(int)

    print("Final threshold:", threshold)
    print("Predicted anomalies:", int(decision.sum()), "/", len(decision))

    # Output files
    anomaly_output = pd.DataFrame({
        "file_id": file_ids,
        "score": test_final_score
    })

    decision_output = pd.DataFrame({
        "file_id": file_ids,
        "decision": decision
    })

    anomaly_file = os.path.join(
        OUTPUT_DIR,
        f"anomaly_score_{machine}_section_00_test.csv"
    )

    decision_file = os.path.join(
        OUTPUT_DIR,
        f"decision_result_{machine}_section_00_test.csv"
    )

    # VERY IMPORTANT: no header, no index
    anomaly_output.to_csv(
        anomaly_file,
        index=False,
        header=False
    )

    decision_output.to_csv(
        decision_file,
        index=False,
        header=False
    )

    # Also save detailed component scores for analysis
    detail_df = pd.DataFrame({
        "file_id": file_ids,
        "final_anomaly_score": test_final_score,
        "decision": decision
    })

    for k, v in all_test_components.items():
        detail_df[k] = v

    detail_file = os.path.join(
        OUTPUT_DIR,
        f"detail_scores_{machine}.csv"
    )

    detail_df.to_csv(detail_file, index=False)

    summary_rows.append({
        "machine": machine,
        "train_file": os.path.basename(train_file),
        "test_file": os.path.basename(test_file),
        "n_train_normal": len(X_train_raw_all),
        "n_test": len(test_df),
        "n_features": len(feature_cols),
        "threshold": threshold,
        "predicted_anomalies": int(decision.sum()),
        "predicted_normals": int(len(decision) - decision.sum()),
        "anomaly_file": os.path.basename(anomaly_file),
        "decision_file": os.path.basename(decision_file),
    })


# ============================================================
# 10. SAVE SUMMARY AND ZIP OUTPUTS
# ============================================================

summary_df = pd.DataFrame(summary_rows)

summary_path = os.path.join(OUTPUT_DIR, "ae_submission_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)
print(summary_df)


# Zip only official output files
zip_path = os.path.join(OUTPUT_DIR, "ae_dcase_submission_files.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:

    for machine in MACHINE_TYPES:

        anomaly_file = os.path.join(
            OUTPUT_DIR,
            f"anomaly_score_{machine}_section_00_test.csv"
        )

        decision_file = os.path.join(
            OUTPUT_DIR,
            f"decision_result_{machine}_section_00_test.csv"
        )

        zf.write(anomaly_file, arcname=os.path.basename(anomaly_file))
        zf.write(decision_file, arcname=os.path.basename(decision_file))

print("\nSaved output directory:")
print(OUTPUT_DIR)

print("\nSaved zip file:")
print(zip_path)

print("\nOfficial files created:")
for machine in MACHINE_TYPES:
    print(f"- anomaly_score_{machine}_section_00_test.csv")
    print(f"- decision_result_{machine}_section_00_test.csv")

TensorFlow: 2.20.0
GPU available:
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

MACHINE: AutoTrash
Train file: /content/cubical_mel_tda_features_AutoTrash_thr.xlsx
Test file : /content/cubical_mel_tda_features_AutoTrash.xlsx
Train shape: (1000, 276)
Test shape : (200, 276)
Feature count: 264
Normal training pool: (1000, 264)
AE normal train: (800, 264)
AE normal val  : (200, 264)
Training compact_sparse_latent4 | latent=4 | noise=0.0 | sparse_l1=1e-05
  best_val_loss=0.427161, epochs=162
Training medium_denoising_latent8 | latent=8 | noise=0.03 | sparse_l1=0.0
  best_val_loss=0.278505, epochs=145
Training wide_sparse_latent32 | latent=32 | noise=0.0 | sparse_l1=1e-05
  best_val_loss=0.183384, epochs=300
Final threshold: 0.9010833333333333
Predicted anomalies: 22 / 200

MACHINE: BandSealer
Train file: /content/cubical_mel_tda_features_BandSealer_thr.xlsx
Test file : /content/cubical_mel_tda_features_BandSealer.xlsx
Train shape: (1000, 276)
Test shape : (200, 276)
Fea